In [36]:
import os

target_folder = "CS6423_knowledge_distillation_project" 
path = os.path.join(os.getcwd(), target_folder)

if not os.getcwd().endswith(target_folder):
    os.chdir(path)

# should match the folder you cloned into
print(f"Current working directory: {os.getcwd()}")

Current working directory: /home/cor10/CS6423_knowledge_distillation_project


In [37]:
# !pip install torch torchvision torchsummary thop

In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune

from torchvision import models
from torch.utils.data import DataLoader

import pandas as pd
from PIL import Image
import os

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# cuda is the GPU, ideally run there
print("Device:", device)

Device: cuda


In [40]:
# import sys to fix file path errors
import sys
sys.path.append('.')

# import python modules for working with model and data
from modules.dataset_prepper import datasetPrepper
from modules.imagenet_loader import ImagenetLoader
from modules.evaluate_model import ModelEvaluator

In [41]:
# prep the dataset
data_prep = datasetPrepper(
    dataframe_path="data/labels.csv", # csv with filenames and labels
    image_dir="data/test_images", # the actual images
    batch_size=32
).prepare(compute_class_weights=True) # compute weights for imbalanced classes
# test_split default = 0.2


train_loader = data_prep.train_loader
val_loader = data_prep.val_loader

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 225
Validation batches: 57


In [42]:
loader = ImagenetLoader()

# we'll use resnet50 as the teacher and resnet18 as the student for this version of the experiment
teacher = loader.load_radimagenet_resnet50(
    "./trained_models/resnet50_baseline/resnet50_baseline.pth",
    load_type="load"
    )

# move teacher model to GPU
teacher = teacher.to(device)
teacher.eval()


# Freeze all teacher parameters so they are not updated during training
# The teacher only provides guidance for the student
for p in teacher.parameters():
    p.requires_grad = False

In [43]:
# the student model architecture is going to be based on the resnet18 model
# student learns from teacher resnet50 model
student = loader.load_radimagenet_resnet18(
    "./trained_models/resnet18_baseline/resnet18_baseline.pth",
    load_type="load"
)

# put student model in gpu
student = student.to(device)

In [44]:
'''
Define the loss used for PWKD
Combines two losses:
1. Cross-Entropy Loss (ce_loss) for normal training
2. Knowledge Distillation loss (kd_loss) where student learns from teacher: uses kl divergence
'''
class PWKDLoss(nn.Module):

    def __init__(self, temperature=4, alpha=0.5):
        super().__init__()
        
        # Temperature controls the softness of probability distributions
        # Higher values produce softer probability outputs
        self.temperature = temperature
        
        # Alpha balances CE loss and KD loss
        # alpha = weight of classification loss
        self.alpha = alpha
        
        # Standard classification loss
        self.ce = nn.CrossEntropyLoss()
        
        # KL divergence for knowledge distillation
        self.kl = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_logits, labels):

        # Std loss with ground truth labels
        ce_loss = self.ce(student_logits, labels)

        # KD loss when comparing teacher and student
        kd_loss = self.kl(
            torch.log_softmax(student_logits / self.temperature, dim=1),
            torch.softmax(teacher_logits / self.temperature, dim=1)
        ) * (self.temperature ** 2)

        # combination: PWKD formula
        return ce_loss * self.alpha + kd_loss * (1 - self.alpha)

In [45]:
# applying magnitude-based pruning to layers
def apply_pruning(model, amount=0.1):

    for module in model.modules():

        # only apply to convulution and/or fully connected layers
        if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):

            prune.l1_unstructured(
                module,
                name="weight",
                amount=amount
            )

In [46]:
# training loop

# optimiser for updating student model
optimiser = torch.optim.Adam(student.parameters(), lr=1e-4)

# set up loss function
loss_function = PWKDLoss(temperature=4, alpha=0.5)

epochs = 10

# begin training
for epoch in range(epochs):

    student.train()

    total_loss = 0

    # iterating through batches
    for images, labels in train_loader:

        # put batch data in gpu
        images = images.to(device)
        labels = labels.to(device)

        # teacher predictions (no gradients, teacher is frozen)
        with torch.no_grad():
            teacher_logits = teacher(images)

        # student predictions
        student_logits = student(images)

        # compute pwkd loss
        loss = loss_function(student_logits, teacher_logits, labels)

        
        optimiser.zero_grad() # reset gradients
        loss.backward() # backpropagate loss
        optimiser.step() # update student model

        total_loss += loss.item()

    print(f"Epoch {epoch+1} loss:", total_loss / len(train_loader))

    # pruning step
    apply_pruning(student, amount=0.05)

Epoch 1 loss: 1.5654835764567057
Epoch 2 loss: 1.1864441580242582
Epoch 3 loss: 1.099092267619239
Epoch 4 loss: 1.0125757116741605
Epoch 5 loss: 0.9549454455905491
Epoch 6 loss: 0.9132084470325046
Epoch 7 loss: 0.8893427965376112
Epoch 8 loss: 0.8566942969957988
Epoch 9 loss: 0.8460241813129848
Epoch 10 loss: 0.8179538859261407


In [47]:
# remove masks from weights to make them permanent
for module in student.modules():

    # check if pruned
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):

        try:
            prune.remove(module, "weight")
        except:
            pass # ignore non-pruned modules

In [48]:
evaluator = ModelEvaluator(
    data_loader=val_loader,
    class_names=data_prep.class_names
)

baseline_model_18 = loader.load_radimagenet_resnet18(
    "./trained_models/resnet18_baseline/resnet18_baseline.pth",
    load_type="load"
    )

#compare all models, teacher, initial student architecture, and final PWKD student
evaluator.evaluate_many({
    "Teacher ResNet50": teacher,
    "Pre-PWKD Student": baseline_model_18,
    "Post-PWKD Student": student
})


[Evaluating] Teacher ResNet50...

Warming up Teacher ResNet50...
Running inference...

[Evaluating] Pre-PWKD Student...

Warming up Pre-PWKD Student...
Running inference...

[Evaluating] Post-PWKD Student...

Warming up Post-PWKD Student...
Running inference...

+-------------------+------------+---------------------+-------------------+
| Model             |   F1 Macro |   cuda Latency (ms) |   Model Size (mb) |
+===================+============+=====================+===================+
| Teacher ResNet50  |       0.84 |                1.1  |             90.83 |
+-------------------+------------+---------------------+-------------------+
| Pre-PWKD Student  |       0.74 |                0.36 |             42.91 |
+-------------------+------------+---------------------+-------------------+
| Post-PWKD Student |       0.99 |                0.36 |             42.91 |
+-------------------+------------+---------------------+-------------------+


In [49]:
import pandas as pd

df = pd.DataFrame(evaluator.results).T

df.head()

,f1_micro,f1_macro,sensitivity,avg_latency_ms,model_size_mb,device
Teacher ResNet50,0.801667,0.843127,0.924454,1.098947,90.832657,cuda
Pre-PWKD Student,0.666667,0.735586,0.869621,0.355341,42.91053,cuda
Post-PWKD Student,0.977222,0.986753,0.995733,0.35528,42.91053,cuda
